# Whole Voronoi cells at the model boundary

`MeshGenerator(boundary_method="conforming")` uses the existing circle-barrier
construction for the active domain's exterior and holes. Temporary sites on
the inactive side make the outline a set of Voronoi faces; whole inactive
cells are discarded. No active polygon is clipped, and numerical centres
remain their Voronoi generators.

The default `boundary_method="clip"` remains available. The two runs below
use the same requested domain and size fields, but build different meshes.
Install this checkout in the notebook kernel with
`%pip install -e ".[examples]"` from the repository root, then restart the
kernel if it previously imported another checkout.

In [ ]:
import numpy as np
import pandas as pd
import shapely
from shapely.geometry import Polygon, box
import matplotlib.pyplot as plt

from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator, set_verbosity

set_verbosity(0)
domain = Polygon([(0, 0), (200, 0), (200, 100), (130, 100),
                  (130, 60), (90, 60), (90, 100), (0, 100)])
domain = domain.difference(box(30, 35, 55, 60))

def make_grid(method):
    cm = ConceptualMesh()
    cm.add_polygon(domain, zone_id="aquifer", resolution=12.)
    # A size field only: this rectangle must not become a new model boundary.
    cm.add_polygon(box(-5, -5, 70, 105), zone_id="refinement",
                   resolution=6., embed=False)
    mg = MeshGenerator(background_lc=12., conceptual_mesh=cm,
                       boundary_method=method)
    mg.generate()
    vt = VoronoiTessellator(mg)
    grid = vt.generate()
    return mg, vt, grid

runs = {method: make_grid(method) for method in ["clip", "conforming"]}


## Compare the complete domain and a reflex corner

Blue dots are actual retained Voronoi generators. Orange crosses in the
conforming close-up are discarded construction generators. These are not
original Gmsh element outlines. The red line is the unchanged requested
active-domain boundary. The size field refines the left side to 6 m and
grades to 12 m on the right through the existing barrier sizing code.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
for row, (method, (mg, vt, grid)) in enumerate(runs.items()):
    for col, ax in enumerate(axes[row]):
        grid.plot(ax=ax, color="#d6e9f5", edgecolor="#526b7d", linewidth=.6)
        ax.scatter(grid.x, grid.y, s=7, color="#195f95", zorder=3)
        if method == "conforming" and col == 1:
            removed = vt.boundary_construction_grid
            removed = removed[removed.node_id.isin(vt.boundary_report["discarded_node_ids"])]
            ax.scatter(removed.x, removed.y, s=17, color="#c06b26", marker="x", zorder=3)
        for polygon in getattr(domain, "geoms", [domain]):
            for ring in [polygon.exterior, *polygon.interiors]:
                xy = np.asarray(ring.coords)
                ax.plot(xy[:, 0], xy[:, 1], color="#c83535", linewidth=1.2, zorder=4)
        ax.set_aspect("equal")
        ax.set_xlabel("x (m)")
        ax.set_ylabel("y (m)")
        if col == 1:
            ax.set_xlim(73, 109)
            ax.set_ylim(43, 79)
        else:
            ax.set_xlim(-3, 203)
            ax.set_ylim(-3, 103)
        ax.set_title(f"{method}: {'complete domain' if col == 0 else 'corner close-up'}")
plt.show()


## Measure the result

Compactness is `4*pi*area/perimeter**2` (one for a circle). Centre clearance
is the distance from the retained generator to its own polygon boundary.
Clipping can leave a generator on the cut face, giving zero clearance.
Conforming construction removes that particular defect; it does not promise
ideal aspect ratios for arbitrarily narrow or sharply pointed inputs.

In [ ]:
measurements = []
for method, (mg, vt, grid) in runs.items():
    boundary = grid.geometry.distance(domain.boundary) < 1e-7
    cells = grid.loc[boundary]
    clearance = shapely.distance(shapely.points(cells.x, cells.y), cells.geometry.boundary)
    measurements.append({
        "method": method,
        "cells": len(grid),
        "boundary cells": len(cells),
        "minimum boundary compactness": float((4*np.pi*cells.area/cells.length**2).min()),
        "minimum centre clearance (m)": float(clearance.min()),
        "concave cells": int(((grid.geometry.convex_hull.area-grid.area) > 1e-8).sum()),
    })

mg, vt, grid = runs["conforming"]
assert vt.boundary_report["cell_clip_area"] == 0
assert vt.n_exact_barrier_mirrors == vt.n_exact_barrier_cuts == 0
assert shapely.union_all(grid.geometry).symmetric_difference(domain).area < 1e-7
assert abs(grid.area.sum() - domain.area) < 1e-7
assert (grid.geometry.convex_hull.area - grid.area).max() < 1e-7
assert np.isfinite(grid[["x", "y"]].to_numpy()).all()
assert set(grid.zone_id) == {"aquifer"}
pd.DataFrame(measurements).set_index("method")


The support region exists only during construction. `mg.zones_gdf` keeps
the physical zones, and `vt.grid` and the DISU/GSF exporters contain only
active cells. `vt.boundary_construction_grid` and `vt.boundary_report`
expose the discarded cells and coverage checks.

Set the option before `mg.generate()`: the perimeter is part of native
barrier planning and field-driven sizing. Leave `boundary_centering` at
its default and `clip_to_boundary=True`; conforming mode selects whole cells
instead of calling the polygon clipper. Unsupported acute corners or
fixed feature centres on the outline raise an error rather than silently
cutting cells. Simplify or refine incompatible geometry, or explicitly use
the legacy clipping method.

Generators are inside the active area, so a head prescribed at a physical
boundary face needs the appropriate boundary conductance. Assigning that
head directly to an interior cell changes its location. The regression
suite checks an MF6 uniform-flow model with face-to-centre GHB
conductances against the analytical solution.